# SmartQ — Model Training & Evaluation

This notebook reproduces the official SmartQ model comparison required by the project proposal.

Models: **Linear Regression, Random Forest, XGBoost**  
Metrics: **MAE and RMSE**  
Selection rule: **lowest validation MAE**.


## 1. Result summary

The official validation results are:

| Model | Validation MAE | Validation RMSE |
|---|---:|---:|
| Linear Regression | 4.1146 | 6.1452 |
| Random Forest | 2.6315 | 4.4920 |
| **XGBoost** | **2.6302** | **4.3893** |

XGBoost has the lowest validation MAE and is therefore selected according to the declared project rule. Random Forest is extremely close, so this should be described as a near-tie rather than a dramatic difference.


## 2. Final unseen test results

After selection was fixed using validation data, the selected XGBoost model was evaluated once on the untouched test set.

| Model | Test MAE | Test RMSE |
|---|---:|---:|
| Linear Regression | 4.0645 | 6.4606 |
| Random Forest | 2.5231 | 4.6425 |
| **Selected XGBoost** | **2.5824** | **4.9561** |
| Mean baseline | 14.9850 | 24.8143 |
| SmartQ deterministic ETA | 4.6386 | 6.3683 |

Random Forest happens to score slightly lower on the final test MAE, but **we do not switch models after looking at the test set**. Doing that would use test performance for model selection and weaken the evaluation methodology. XGBoost remains the selected model because it won on validation MAE.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Run from repository root or notebooks/.
DATASET_NAME = 'SmartQ_Synthetic_Operational_Dataset_100k.csv'
candidate_paths = [Path('data') / DATASET_NAME, Path('..') / 'data' / DATASET_NAME]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('SmartQ dataset not found.')

df = pd.read_csv(data_path, parse_dates=['scenario_date'])
completed = df[df['status'] == 'COMPLETED'].copy()


In [ ]:
TARGET = 'actual_wait_minutes'
numeric_features = [
    'arrival_offset_minutes','people_ahead','general_waiting','priority_waiting',
    'serving_count','open_general_counters','open_priority_counters','effective_open_counters',
    'counter_utilisation','queue_pressure_index','workload_minutes_ahead',
    'recent_avg_service_minutes_10','recent_avg_wait_minutes_10','recent_throughput_60m',
    'service_target_minutes','hour_of_day'
]
categorical_features = ['branch_code','service_code','booking_source','queue_type','day_of_week','is_peak_period']
features = numeric_features + categorical_features

dates = sorted(completed['scenario_date'].dt.normalize().unique())
n = len(dates)
train_end = pd.Timestamp(dates[int(n*0.70)-1])
val_start = pd.Timestamp(dates[int(n*0.70)])
val_end = pd.Timestamp(dates[int(n*0.85)-1])
test_start = pd.Timestamp(dates[int(n*0.85)])

train = completed[completed['scenario_date'] <= train_end].copy()
validation = completed[(completed['scenario_date'] >= val_start) & (completed['scenario_date'] <= val_end)].copy()
test = completed[completed['scenario_date'] >= test_start].copy()

X_train,y_train = train[features],train[TARGET]
X_val,y_val = validation[features],validation[TARGET]
X_test,y_test = test[features],test[TARGET]

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features),
])
Xtr = preprocessor.fit_transform(X_train)
Xv = preprocessor.transform(X_val)
Xt = preprocessor.transform(X_test)


## 3. Baselines

The project requires a simple mean-wait baseline. The existing deterministic SmartQ ETA is also reported as an extra engineering comparison, but it is not used as an input feature to the official ML models.


In [ ]:
def score(y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred), 0, None)
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
    }

mean_wait = y_train.mean()
print('Mean baseline:', score(y_val, np.full(len(y_val), mean_wait)))
print('Deterministic ETA:', score(y_val, validation['baseline_eta_minutes']))


## 4. Linear Regression


In [ ]:
linear = LinearRegression()
linear.fit(Xtr, y_train)
linear_val_pred = np.clip(linear.predict(Xv), 0, None)
print(score(y_val, linear_val_pred))


## 5. Random Forest — limited tuning

Two deliberately small parameter combinations are compared. This keeps tuning understandable and within the proposal scope.


In [ ]:
rf_candidates = [
    {'n_estimators':100,'max_depth':18,'min_samples_leaf':2,'max_features':0.8},
    {'n_estimators':150,'max_depth':14,'min_samples_leaf':2,'max_features':1.0},
]

rf_trials = []
for params in rf_candidates:
    model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    model.fit(Xtr, y_train)
    pred = np.clip(model.predict(Xv), 0, None)
    rf_trials.append({**params, **score(y_val, pred)})
display(pd.DataFrame(rf_trials))


## 6. XGBoost — limited tuning


In [ ]:
xgb_candidates = [
    {'n_estimators':150,'max_depth':4,'learning_rate':0.05,'subsample':0.9,'colsample_bytree':0.9,'reg_lambda':1.0},
    {'n_estimators':200,'max_depth':5,'learning_rate':0.08,'subsample':0.9,'colsample_bytree':0.9,'reg_lambda':2.0},
]

xgb_trials = []
for params in xgb_candidates:
    model = XGBRegressor(random_state=42, n_jobs=-1, objective='reg:squarederror', tree_method='hist', **params)
    model.fit(Xtr, y_train)
    pred = np.clip(model.predict(Xv), 0, None)
    xgb_trials.append({**params, **score(y_val, pred)})
display(pd.DataFrame(xgb_trials))


## 7. Selected model and traffic-scenario performance

XGBoost is selected by validation MAE. On the final test set its MAE is **2.5824 minutes** and RMSE is **4.9561 minutes**.

Traffic scenarios are defined from `queue_pressure_index` for reporting:

- Low: < 1.0
- Moderate: 1.0 to < 2.5
- Busy: >= 2.5

Test MAE rises from about **1.22 minutes in Low traffic** to **7.94 minutes in Busy traffic**, which is an important limitation to discuss rather than hide.


## 8. Conclusion

The three required models were trained on the same chronologically separated data and evaluated with the same MAE/RMSE metrics. XGBoost achieved the lowest validation MAE and therefore becomes the SmartQ integration candidate. The final test set remains a one-time evaluation set and is not used to re-select the model.
